### **ASPEKT Paper Reviewer**


In [ ]:
import json
import os
import copy
import re
import unicodedata
import shutil
import tempfile
from pathlib import Path
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill


# ── Tenta importar findpapers para download de PDF ──────────
try:
    import findpapers
    HAS_FINDPAPERS = True
except ImportError:
    HAS_FINDPAPERS = False
    print("⚠️  findpapers não encontrado. Download de PDFs desativado.")
    print("   Para ativar: pip install findpapers")

print("✅ Dependências carregadas.")

### **Configurações**

In [ ]:
# CONFIGURAÇÕES — edite conforme necessário
# ============================================================
 
INPUT_JSON   = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados_novos\unificado_findpapers_dedup.json"      # Caminho para o JSON do findpapers
OUTPUT_JSON  = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados_novos\selected_papers.json"      # JSON consolidado dos artigos ACEITOS
STATE_JSON   = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados_novos\classification_state.json"  # Estado COMPLETO (aceitos + rejeitados) — usado para restaurar a sessão
PDF_DIR      = r"..//data//artigos//pdfs"                      # Pasta raiz onde as subpastas por caixa serão criadas

In [ ]:
# CAIXAS DO ASPEKT
# ============================================================

ASPEKT_BOXES = [
    # ── Pré-processamento / Setup ──────────────────────────
    "1. VFSS Recording & Segmentation",
    "2. Bolus Count & Subswallow Detection",

    # ── Escala de Penetração-Aspiração ────────────────────
    "3. Penetration-Aspiration Scale (PAS)",

    # ── Eventos Faríngeos ─────────────────────────────────
    "4a. Peak XT / Hyoid Tracking (velocity, position)",
    "4b. Hyoid Burst (BPM, onset/offset)",
    "4c. Laryngeal Vestibule Closure (LVC / IVA)",
    "4d. UES Opening & Maximum UES Distension",
    "4e. Maximum Pharyngeal Constriction (MPC)",
    "4f. Swallow Rest / UES Closure (UESC)",

    # ── Métricas Espaciais / Morfológicas ─────────────────
    "5a. Bolus Location & Tracking",
    "5b. UES Diameter & MPA Area",
    "5c. Pharyngeal Area at Rest",
    "5d. Normalized Residue Scale (NRS)",

    # ── Opções especiais ──────────────────────────────────
    "Múltiplas caixas",
    "Caixa não identificada",
]

# Motivos de rejeição disponíveis
REJECT_REASONS = [
    "Tipo de dado incompatível",
    "Área de estudo incompatível",
    "Review",
    "Pouco relevante",
    "Pesquisa do Som",
    "UTIL - Para ver depois"
]

print(f"✅ {len(ASPEKT_BOXES)} caixas ASPEKT configuradas.")

In [ ]:
# CARREGAMENTO DO JSON
# ============================================================

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

papers = raw_data.get("papers", [])
print(f"✅ {len(papers)} artigos carregados de '{INPUT_JSON}'")

# Estado da sessão
# {index: {"status": "aceito"/"rejeitado", "box": ..., "reason": ..., "downloaded": ..., "download_info": ...}}
selected = {}

title_to_idx = {p["title"]: i for i, p in enumerate(papers) if p.get("title")}

if os.path.exists(STATE_JSON):
    # Caminho normal: restaura tudo (aceitos e rejeitados) do arquivo de estado completo
    with open(STATE_JSON, "r", encoding="utf-8") as f:
        prev_state = json.load(f)
    restored = 0
    for title, info in prev_state.items():
        if title in title_to_idx:
            selected[title_to_idx[title]] = info
            restored += 1
    print(f"   ↩️  {restored} classificações anteriores restauradas de '{STATE_JSON}'")

elif os.path.exists(OUTPUT_JSON):
    # Fallback de compatibilidade com sessões antigas (antes do arquivo de estado existir):
    # só recupera os ACEITOS, pois os rejeitados nunca foram salvos em disco nesse formato.
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        prev = json.load(f)
    prev_papers = prev.get("papers", [])
    restored = 0
    for p in prev_papers:
        if p.get("title") in title_to_idx:
            selected[title_to_idx[p["title"]]] = {"status": "aceito", "box": p.get("aspekt_box")}
            restored += 1
    print(f"   ↩️  {restored} seleções (apenas aceitos) restauradas de '{OUTPUT_JSON}'.")
    print(f"       A partir de agora, '{STATE_JSON}' vai guardar tudo (aceitos + rejeitados).")

else:
    print("   ℹ️  Nenhum estado anterior encontrado — começando do zero.")

In [ ]:
# FUNÇÕES AUXILIARES
# ============================================================

# Caminho do Excel: mesmo diretório/nome do OUTPUT_JSON, com extensão .xlsx
OUTPUT_XLSX = Path(OUTPUT_JSON).with_suffix(".xlsx")


def save_output():
    """Salva três arquivos, sempre em conjunto:

    1. OUTPUT_JSON  — apenas os artigos ACEITOS (formato de trabalho, usado
       no fluxo de download de PDF).
    2. STATE_JSON   — TODOS os artigos classificados (aceitos e rejeitados,
       com motivo). É esse arquivo que restaura a sessão caso o notebook
       seja reiniciado — nada de classificação é perdido.
    3. OUTPUT_XLSX  — planilha com TODOS os artigos (aceitos, rejeitados e
       não avaliados), status, motivo e situação do download.
    """
    out = copy.deepcopy(raw_data)
    out["papers"] = []

    # Filtra apenas os itens com status "aceito"
    # (compatível também com o formato antigo, onde o valor era
    #  direto uma string com o nome da caixa = sempre "aceito")
    accepted_items = {
        idx: info for idx, info in selected.items()
        if (info.get("status") if isinstance(info, dict) else "aceito") == "aceito"
    }

    out["number_of_papers"] = len(accepted_items)
    for idx, info in accepted_items.items():
        box = info.get("box") if isinstance(info, dict) else info
        paper = copy.deepcopy(papers[idx])
        paper["aspekt_box"] = box
        paper["selected"] = True
        out["papers"].append(paper)

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    save_state()
    export_xlsx()


def save_state():
    """Salva o estado COMPLETO de classificação (aceitos e rejeitados,
    com motivo, status de download etc.) em STATE_JSON, indexado pelo
    título do artigo. É esse arquivo que é lido no início do notebook
    para restaurar a sessão sem perder nenhuma classificação.
    """
    state = {}
    for idx, info in selected.items():
        title = papers[idx].get("title")
        if not title:
            continue
        # normaliza o formato antigo (string) para dict, se necessário
        state[title] = info if isinstance(info, dict) else {"status": "aceito", "box": info}

    with open(STATE_JSON, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def export_xlsx():
    """Gera um xlsx com TODOS os artigos (não só os selecionados),
    incluindo os mesmos metadados do JSON como colunas, mais:
    Status (Aceito / Rejeitado / Não avaliado), Caixa ASPEKT,
    Motivo da Rejeição e se o PDF foi baixado (só se aplica a aceitos).
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "Artigos"

    headers = [
        "Título", "Autores", "Data de Publicação", "Publicação", "Categoria",
        "DOI", "Bases de Dados", "Palavras-chave", "Abstract", "URLs",
        "Status", "Caixa ASPEKT", "Motivo da Rejeição", "PDF Baixado",
    ]
    ws.append(headers)

    header_font = Font(name="Arial", bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="1E40AF", end_color="1E40AF", fill_type="solid")
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"

    body_font = Font(name="Arial")

    for idx, p in enumerate(papers):
        info = selected.get(idx)
        status_raw = info.get("status") if isinstance(info, dict) else ("aceito" if info else None)

        box = ""
        reason = ""
        download_label = "N/A"

        if status_raw == "aceito":
            status_label = "Aceito"
            box = info.get("box", "") if isinstance(info, dict) else info
            downloaded = info.get("downloaded") if isinstance(info, dict) else None
            if downloaded is True:
                download_label = "Sim"
            elif downloaded is False:
                download_label = "Não"
            else:
                download_label = "—"  # artigo aceito no formato antigo, sem essa info registrada
        elif status_raw == "rejeitado":
            status_label = "Rejeitado"
            reason = info.get("reason", "")
        else:
            status_label = "Não avaliado"

        pub = p.get("publication") or {}
        row = [
            p.get("title") or "",
            "; ".join(p.get("authors") or []),
            p.get("publication_date") or "",
            pub.get("title") or "",
            pub.get("category") or "",
            p.get("doi") or "",
            ", ".join(p.get("databases") or []),
            ", ".join(p.get("keywords") or []),
            p.get("abstract") or "",
            ", ".join(p.get("urls") or []),
            status_label,
            box,
            reason,
            download_label,
        ]
        ws.append(row)
        r = ws.max_row
        for col_idx in range(1, len(headers) + 1):
            cell = ws.cell(row=r, column=col_idx)
            cell.font = body_font
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    # Larguras de coluna aproximadas
    widths = [40, 25, 14, 25, 16, 18, 18, 30, 60, 30, 12, 18, 22, 12]
    for col_idx, w in enumerate(widths, start=1):
        ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = w

    wb.save(OUTPUT_XLSX)


def download_pdf(paper, box_name):
    """Tenta baixar o PDF do artigo usando findpapers.download."""
    if not HAS_FINDPAPERS:
        return False, "findpapers não instalado"

    # Pasta destino
    safe_box = "".join(c if c.isalnum() or c in " _-" else "_" for c in box_name).strip()
    dest_dir = Path(PDF_DIR) / safe_box
    dest_dir.mkdir(parents=True, exist_ok=True)

    # ── Copia e sanitiza os campos críticos para o downloader ──
    p = copy.deepcopy(paper)

    # urls: precisa ser uma lista (nunca None)
    urls = p.get("urls") or []
    if isinstance(urls, str):          # caso venha como string única
        urls = [urls]
    p["urls"] = list(urls)

    # doi: se existir, adiciona como URL de fallback (o downloader faz isso internamente,
    # mas garantimos que o campo esteja presente e bem-formado)
    doi = p.get("doi")
    if doi and f"http://doi.org/{doi}" not in p["urls"]:
        p["urls"].append(f"http://doi.org/{doi}")

    # keywords e databases também precisam ser listas (não None) para o from_dict não quebrar
    p["keywords"]  = list(p.get("keywords")  or [])
    p["databases"] = list(p.get("databases") or [])

    # Marca como selecionado
    p["selected"] = True

    # Monta JSON temporário
    tmp_data = copy.deepcopy(raw_data)
    tmp_data["papers"] = [p]
    tmp_data["number_of_papers"] = 1

    # Diagnóstico rápido: avisa no log se ainda não houver URLs
    if not p["urls"]:
        return False, f"Artigo sem URLs e sem DOI — impossível baixar: '{p.get('title')}'"

    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".json", delete=False, encoding="utf-8"
        ) as tf:
            json.dump(tmp_data, tf, ensure_ascii=False)
            tmp_path = tf.name

        findpapers.download(
            search_path=tmp_path,
            output_directory=str(dest_dir),
            only_selected_papers=True,
        )
        os.unlink(tmp_path)
        return True, str(dest_dir)
    except Exception as e:
        if tmp_path:
            try:
                os.unlink(tmp_path)
            except Exception:
                pass
        return False, str(e)


def fmt_authors(authors):
    if not authors:
        return "—"
    if len(authors) <= 3:
        return "; ".join(authors)
    return "; ".join(authors[:3]) + f" … (+{len(authors)-3})"


def fmt_keywords(keywords):
    if not keywords:
        return "—"
    clean = [k.lstrip("N ").strip() for k in keywords]
    return " · ".join(clean)


def fmt_urls(urls):
    if not urls:
        return "—"
    links = [f'<a href="{u}" target="_blank">{u}</a>' for u in urls]
    return "<br>".join(links)

print("✅ Funções auxiliares prontas.")

In [ ]:
# MIGRAÇÃO DE CLASSIFICAÇÕES DE UMA QUERY ANTERIOR
# ============================================================
# Rode esta célula (e a função abaixo) DEPOIS de "Funções Auxiliares"
# e DEPOIS do "Carregamento do JSON" (que já carrega a query NOVA em
# `papers`), e ANTES de abrir a "Interface Principal".


# ------------------------------------------------------------
# CAMINHOS — EDITE CONFORME NECESSÁRIO
# ------------------------------------------------------------
# Os arquivos da query ANTIGA têm os MESMOS NOMES dos da query atual
# (INPUT_JSON / STATE_JSON, definidos na célula de Configurações),
# só ficam em uma pasta diferente. Basta editar OLD_QUERY_DIR abaixo.

OLD_QUERY_DIR = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados"  # <-- pasta da query antiga

OLD_INPUT_JSON = os.path.join(OLD_QUERY_DIR, os.path.basename(INPUT_JSON))  # metadados brutos da query antiga
OLD_STATE_JSON = os.path.join(OLD_QUERY_DIR, os.path.basename(STATE_JSON))  # classificações da query antiga

print(f"📂 INPUT_JSON antigo: {OLD_INPUT_JSON}")
print(f"📂 STATE_JSON antigo: {OLD_STATE_JSON}")
print(f"📂 INPUT_JSON novo:   {INPUT_JSON}")
print(f"📂 STATE_JSON novo:   {STATE_JSON}")


def _normalize_title(title):
    """Normaliza um título pra comparação: sem acento, minúsculo, sem pontuação."""
    if not title:
        return ""
    t = unicodedata.normalize("NFKD", title)
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = t.lower()
    t = re.sub(r"[^a-z0-9]+", " ", t)
    return t.strip()


def _normalize_doi(doi):
    """Normaliza um DOI pra comparação: sem prefixo de URL, minúsculo, sem barra final."""
    if not doi:
        return ""
    d = doi.strip().lower()
    d = re.sub(r"^(https?://)?(dx\.)?doi\.org/", "", d)
    return d.rstrip("/")


def migrate_from_previous_query(old_input_json_path, old_state_json_path, overwrite=False):
    """Aproveita classificações (status, caixa ASPEKT, motivo de rejeição,
    status de download etc.) feitas numa query ANTERIOR e aplica na query
    ATUAL (já carregada em `papers`), casando os artigos por DOI (prioridade)
    e, quando não houver DOI, por título normalizado.

    old_input_json_path : caminho do INPUT_JSON (findpapers) da query antiga
                           — usado só para descobrir o DOI de cada título antigo.
    old_state_json_path : caminho do classification_state.json da query antiga
                           — de onde vêm as classificações em si.
    overwrite            : se True, sobrescreve classificações que o artigo já
                            tenha na sessão atual. Se False (padrão), só
                            preenche artigos ainda sem classificação.
    """
    # 1) Metadados da query antiga — só pra conseguir o DOI de cada título
    with open(old_input_json_path, "r", encoding="utf-8") as f:
        old_raw = json.load(f)
    old_papers = old_raw.get("papers", [])
    old_title_to_doi = {
        _normalize_title(p.get("title")): _normalize_doi(p.get("doi"))
        for p in old_papers if p.get("title")
    }

    # 2) Estado de classificação da query antiga (aceitos + rejeitados)
    with open(old_state_json_path, "r", encoding="utf-8") as f:
        old_state = json.load(f)  # {titulo_antigo: info}

    # 3) Índices de busca a partir do estado antigo
    old_title_norm_to_info = {}
    for title, info in old_state.items():
        old_title_norm_to_info[_normalize_title(title)] = info

    # 4) Índices dos artigos da query NOVA (já carregados em `papers`)
    new_doi_to_idx = {}
    new_title_norm_to_idx = {}
    for i, p in enumerate(papers):
        doi = _normalize_doi(p.get("doi"))
        if doi:
            new_doi_to_idx[doi] = i
        norm_title = _normalize_title(p.get("title"))
        if norm_title:
            new_title_norm_to_idx[norm_title] = i

    # 5) Match: DOI primeiro, título normalizado como plano B
    matched_doi = 0
    matched_title = 0
    used_old_keys = set()
    skipped_existing = 0

    for norm_title, info in old_title_norm_to_info.items():
        doi = old_title_to_doi.get(norm_title, "")
        idx = None

        if doi and doi in new_doi_to_idx:
            idx = new_doi_to_idx[doi]
            matched_doi += 1
        elif norm_title in new_title_norm_to_idx:
            idx = new_title_norm_to_idx[norm_title]
            matched_title += 1

        if idx is not None:
            if idx in selected and not overwrite:
                skipped_existing += 1
                continue
            selected[idx] = (
                copy.deepcopy(info) if isinstance(info, dict) else {"status": "aceito", "box": info}
            )
            used_old_keys.add(norm_title)

    not_found_old = len(old_title_norm_to_info) - len(used_old_keys) - skipped_existing
    unmatched_new = sum(1 for i in range(len(papers)) if i not in selected)

    print("═" * 60)
    print("📋 RELATÓRIO DE MIGRAÇÃO")
    print("═" * 60)
    print(f"✔ Migrados por DOI:                               {matched_doi}")
    print(f"✔ Migrados por título (sem DOI compatível):       {matched_title}")
    print(f"↩️  Já classificados nesta sessão (não sobrescritos): {skipped_existing}")
    print(f"⚠ Artigos antigos não encontrados na query nova:   {not_found_old}")
    print(f"ℹ Artigos da query nova sem classificação prévia:  {unmatched_new}")
    print("═" * 60)

    save_output()
    print(f"💾 Migração aplicada e salva em '{OUTPUT_JSON}', '{STATE_JSON}' e '{OUTPUT_XLSX}'.")


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
# Usa os caminhos explícitos definidos no topo do arquivo.
# Troque overwrite=True se quiser reaplicar por cima de classificações
# que você já tenha feito manualmente nesta sessão da query nova.

if False:
    migrate_from_previous_query(
        old_input_json_path=OLD_INPUT_JSON,
        old_state_json_path=OLD_STATE_JSON,
        overwrite=False,
    )

### **Aplição**

In [ ]:
# INTERFACE PRINCIPAL — layout em duas colunas
# ============================================================

state = {"idx": 0, "panel_open": False}

# ── Ordenação de exibição ─────────────────────────────────────
# Mostra primeiro os artigos que AINDA NÃO foram classificados,
# depois os que já foram classificados (aceitos ou rejeitados).
# Calculada uma única vez ao carregar a interface: fica fixa durante
# a sessão (não reordena sozinha quando você classifica um artigo).
# `order[pos]` = índice real do artigo (em `papers`/`selected`) que
# aparece na posição `pos` da navegação.
def _compute_display_order():
    unclassified = [i for i in range(len(papers)) if i not in selected]
    classified   = [i for i in range(len(papers)) if i in selected]
    return unclassified + classified

order = _compute_display_order()

PURPLE = "#7c3aed"  # cor do botão "Desmarcar"

# ── Barra de navegação ───────────────────────────────────────
btn_prev   = widgets.Button(description="◀ Anterior",      button_style="",
                            layout=widgets.Layout(width="140px", height="38px"))
btn_next   = widgets.Button(description="Próximo ▶",       button_style="primary",
                            layout=widgets.Layout(width="140px", height="38px"))
btn_select = widgets.Button(description="★ Selecionar", button_style="success",
                            layout=widgets.Layout(width="175px", height="38px"))
counter_lbl        = widgets.Label()
selected_cnt_lbl   = widgets.HTML()
classified_cnt_lbl = widgets.HTML()

nav_bar = widgets.HBox(
    [btn_prev, counter_lbl, btn_next,
     widgets.HTML("&nbsp;&nbsp;"),
     btn_select, selected_cnt_lbl,
     widgets.HTML("&nbsp;&nbsp;"),
     classified_cnt_lbl],
    layout=widgets.Layout(align_items="center", gap="8px", padding="8px 0"),
)

# ── Coluna esquerda: conteúdo do artigo ─────────────────────
content_out = widgets.Output(
    layout=widgets.Layout(width="100%")
)

# ── Coluna direita: painel de classificação ASPEKT ───────────
panel_out = widgets.Output(
    layout=widgets.Layout(
        width="310px",
        min_width="310px",
        border="2px solid #e2e8f0",
        border_radius="10px",
        padding="0px",
        overflow_y="auto",
    )
)

# Painel começa oculto
panel_out.layout.display = "none"

# ── Layout de duas colunas ───────────────────────────────────
body_row = widgets.HBox(
    [content_out, panel_out],
    layout=widgets.Layout(gap="16px", align_items="flex-start", width="100%"),
)

# ── Log de ações ────────────────────────────────────────────
log_out = widgets.Output(layout=widgets.Layout(height="60px", overflow_y="auto"))

def log(msg, color="#555"):
    ts = datetime.now().strftime("%H:%M:%S")
    with log_out:
        display(HTML(f'<span style="color:{color};font-size:12px">[{ts}] {msg}</span>'))


# ── Helpers para ler o dicionário `selected` de forma segura ─
# (aceita tanto o formato antigo — string com o nome da caixa —
#  quanto o novo formato — dict com status/box/reason)
def _get_status(info):
    if isinstance(info, dict):
        return info.get("status", "aceito")
    return "aceito"  # formato antigo = sempre era uma classificação aceita

def _get_box(info):
    if isinstance(info, dict):
        return info.get("box", "")
    return info  # formato antigo: a própria string já era o nome da caixa

def _get_reason(info):
    if isinstance(info, dict):
        return info.get("reason", "")
    return ""


# ── Renderização do artigo (coluna esquerda) ─────────────────
BADGE_SEL   = '<span style="background:#22c55e;color:#fff;border-radius:4px;padding:2px 10px;font-size:12px;font-weight:700">✔ SELECIONADO</span>'
BADGE_REJ   = '<span style="background:#dc2626;color:#fff;border-radius:4px;padding:2px 10px;font-size:12px;font-weight:700">✖ REJEITADO</span>'
BADGE_UNSEL = '<span style="background:#e5e7eb;color:#888;border-radius:4px;padding:2px 10px;font-size:12px">não selecionado</span>'

def render_paper():
    pos  = state["idx"]        # posição na ordem de exibição
    idx  = order[pos]          # índice real do artigo em `papers`/`selected`
    p    = papers[idx]
    is_sel = idx in selected

    counter_lbl.value = f"{pos + 1} / {len(papers)}"

    # conta apenas os ACEITOS no contador da barra superior
    n_sel = sum(1 for v in selected.values() if _get_status(v) == "aceito")
    selected_cnt_lbl.value = (
        f'<span style="color:#16a34a;font-weight:600">✔ {n_sel} selecionado(s)</span>'
    )

    # conta TODOS os classificados (aceitos + rejeitados), sobre o total de artigos
    n_classified = len(selected)
    classified_cnt_lbl.value = (
        f'<span style="color:#2563eb;font-weight:600">📊 {n_classified}/{len(papers)} classificado(s)</span>'
    )

    if is_sel:
        btn_select.description  = "✖ Desmarcar"
        btn_select.button_style = ""
        btn_select.style.button_color = PURPLE
    else:
        btn_select.description  = "★ Selecionar"
        btn_select.button_style = "success"
        btn_select.style.button_color = None

    if is_sel:
        info   = selected.get(idx)
        status = _get_status(info)
        if status == "rejeitado":
            reason = _get_reason(info)
            status_html = (
                BADGE_REJ +
                f' <span style="background:#fee2e2;color:#991b1b;border-radius:4px;'
                f'padding:2px 10px;font-size:12px">📝 {reason}</span>'
            )
        else:
            box_label = _get_box(info)
            status_html = (
                BADGE_SEL +
                f' <span style="background:#dbeafe;color:#1e40af;border-radius:4px;'
                f'padding:2px 10px;font-size:12px">📂 {box_label}</span>'
            )
    else:
        status_html = BADGE_UNSEL

    pub_info = ""
    if p.get("publication"):
        pub = p["publication"]
        pub_info = pub.get("title") or ""
        if pub.get("category"):
            pub_info += f" ({pub['category']})"

    doi_html = (f'<a href="https://doi.org/{p["doi"]}" target="_blank">{p["doi"]}</a>'
                if p.get("doi") else "—")

    html = f"""
    <div style="font-family:'Segoe UI',sans-serif;padding:0 4px">
      <div style="margin-bottom:12px">{status_html}</div>
      <h2 style="margin:0 0 8px;font-size:1.2rem;color:#1e293b;line-height:1.4">
        {p.get('title') or '(sem título)'}
      </h2>
      <div style="font-size:13px;color:#64748b;margin-bottom:16px;display:flex;flex-wrap:wrap;gap:12px">
        <span>👥 {fmt_authors(p.get('authors'))}</span>
        <span>📅 {p.get('publication_date') or '—'}</span>
        <span>🏛 {pub_info or '—'}</span>
        <span>🔗 DOI: {doi_html}</span>
        <span>🗄 {', '.join(p.get('databases') or []) or '—'}</span>
      </div>
      <div style="margin-bottom:14px">
        <span style="font-size:12px;font-weight:600;color:#475569;text-transform:uppercase;letter-spacing:.05em">Palavras-chave</span><br>
        <span style="font-size:13px;color:#334155">{fmt_keywords(p.get('keywords'))}</span>
      </div>
      <div style="margin-bottom:14px">
        <span style="font-size:12px;font-weight:600;color:#475569;text-transform:uppercase;letter-spacing:.05em">Abstract</span>
        <div style="font-size:13.5px;color:#1e293b;line-height:1.65;margin-top:4px;
                    background:#f8fafc;border-left:3px solid #94a3b8;
                    padding:10px 14px;border-radius:0 6px 6px 0;
                    max-height:340px;overflow-y:auto">
          {p.get('abstract') or '<em>Abstract não disponível.</em>'}
        </div>
      </div>
      <div style="font-size:12px;color:#64748b">
        <strong>URLs:</strong> {fmt_urls(p.get('urls'))}
      </div>
    </div>
    """

    with content_out:
        clear_output(wait=True)
        display(HTML(html))


# ── Painel lateral — ETAPA 1: escolher caixa ASPEKT ou rejeitar ─
def show_panel(idx, pos):
    """Abre o painel lateral com as caixas ASPEKT ao lado do abstract.

    idx : índice real do artigo (em `papers`/`selected`)
    pos : posição do artigo na ordem de exibição atual (para os rótulos)
    """
    state["panel_open"] = True
    panel_out.layout.display = ""

    info = selected.get(idx)
    current_box = _get_box(info) if _get_status(info) == "aceito" else None

    panel_title = widgets.HTML(
        '<div style="background:#1e40af;color:#fff;padding:10px 14px;'
        'border-radius:8px 8px 0 0;font-family:sans-serif">'
        '<div style="font-size:13px;font-weight:700">📂 Caixa ASPEKT</div>'
        f'<div style="font-size:11px;opacity:.8;margin-top:2px">Artigo #{pos+1}</div>'
        '</div>'
    )

    box_select = widgets.RadioButtons(
        options=ASPEKT_BOXES,
        value=current_box if current_box in ASPEKT_BOXES else ASPEKT_BOXES[0],
        layout=widgets.Layout(width="100%"),
    )

    btn_accept = widgets.Button(description="✔ Aceitar", button_style="success",
                                 layout=widgets.Layout(width="110px", height="34px"))
    btn_reject = widgets.Button(description="✖ Rejeitar", button_style="danger",
                                 layout=widgets.Layout(width="110px", height="34px"))
    btn_cancel = widgets.Button(description="✖ Cancelar", button_style="warning",
                                 layout=widgets.Layout(width="110px", height="34px"))
    btn_row = widgets.HBox(
        [btn_accept, btn_reject, btn_cancel],
        layout=widgets.Layout(gap="8px", padding="10px 14px", justify_content="center",
                               flex_flow="row wrap"),
    )

    radio_container = widgets.VBox(
        [box_select],
        layout=widgets.Layout(padding="8px 14px"),
    )

    panel_body = widgets.VBox(
        [panel_title, radio_container, btn_row],
        layout=widgets.Layout(width="100%"),
    )

    def on_accept(_):
        chosen = box_select.value
        ok, dl_info = download_pdf(papers[idx], chosen)
        selected[idx] = {
            "status": "aceito",
            "box": chosen,
            "downloaded": ok,
            "download_info": dl_info,
        }
        save_output()  # salva o JSON (só aceitos) e o xlsx (todos os artigos)
        if ok:
            log(f"📥 PDF salvo em: {dl_info}", "#16a34a")
        elif HAS_FINDPAPERS:
            log(f"⚠️  Não foi possível baixar o PDF: {dl_info}", "#b45309")
        log(f"✔ Artigo #{pos+1} aceito → '{chosen}' | JSON salvo.", "#1d4ed8")
        close_panel()
        render_paper()

    def on_cancel(_):
        log("↩️  Classificação cancelada.", "#64748b")
        close_panel()

    def on_reject_click(_):
        # troca para a etapa 2: escolher o motivo da rejeição
        show_reject_panel(idx, pos)

    btn_accept.on_click(on_accept)
    btn_reject.on_click(on_reject_click)
    btn_cancel.on_click(on_cancel)

    with panel_out:
        clear_output(wait=True)
        display(panel_body)


# ── Painel lateral — ETAPA 2: escolher o motivo da rejeição ──
def show_reject_panel(idx, pos):
    """Segunda etapa do painel lateral: motivo da rejeição.

    idx : índice real do artigo (em `papers`/`selected`)
    pos : posição do artigo na ordem de exibição atual (para os rótulos)
    """
    info = selected.get(idx)
    current_reason = _get_reason(info) if _get_status(info) == "rejeitado" else None

    panel_title = widgets.HTML(
        '<div style="background:#991b1b;color:#fff;padding:10px 14px;'
        'border-radius:8px 8px 0 0;font-family:sans-serif">'
        '<div style="font-size:13px;font-weight:700">🚫 Motivo da rejeição</div>'
        f'<div style="font-size:11px;opacity:.8;margin-top:2px">Artigo #{pos+1}</div>'
        '</div>'
    )

    reason_select = widgets.RadioButtons(
        options=REJECT_REASONS,
        value=current_reason if current_reason in REJECT_REASONS else REJECT_REASONS[0],
        layout=widgets.Layout(width="100%"),
    )

    btn_confirm = widgets.Button(description="✔ Confirmar rejeição", button_style="danger",
                                  layout=widgets.Layout(width="180px", height="34px"))
    btn_back = widgets.Button(description="✖ Cancelar", button_style="warning",
                               layout=widgets.Layout(width="110px", height="34px"))

    btn_row = widgets.HBox(
        [btn_confirm, btn_back],
        layout=widgets.Layout(gap="8px", padding="10px 14px", justify_content="center",
                               flex_flow="row wrap"),
    )

    radio_container = widgets.VBox(
        [reason_select],
        layout=widgets.Layout(padding="8px 14px"),
    )

    panel_body = widgets.VBox(
        [panel_title, radio_container, btn_row],
        layout=widgets.Layout(width="100%"),
    )

    def on_confirm_reject(_):
        reason = reason_select.value
        selected[idx] = {"status": "rejeitado", "reason": reason}
        save_output()
        # PDF não é baixado para artigos rejeitados
        log(f"✖ Artigo #{pos+1} rejeitado → '{reason}' | JSON salvo.", "#dc2626")
        close_panel()
        render_paper()

    def on_back(_):
        log("↩️  Rejeição cancelada.", "#64748b")
        close_panel()

    btn_confirm.on_click(on_confirm_reject)
    btn_back.on_click(on_back)

    with panel_out:
        clear_output(wait=True)
        display(panel_body)


def close_panel():
    state["panel_open"] = False
    panel_out.layout.display = "none"
    with panel_out:
        clear_output(wait=True)


# ── Callbacks dos botões de navegação ────────────────────────
def on_prev(_):
    if state["idx"] > 0:
        state["idx"] -= 1
        close_panel()
        render_paper()

def on_next(_):
    if state["idx"] < len(papers) - 1:
        state["idx"] += 1
        close_panel()
        render_paper()

def on_select(_):
    pos = state["idx"]
    idx = order[pos]
    if idx in selected:
        # Já classificado (aceito ou rejeitado) → desmarcar diretamente
        del selected[idx]
        save_output()
        log(f"✖ Artigo #{pos+1} desmarcado.", "#dc2626")
        close_panel()
        render_paper()
    elif state["panel_open"]:
        # Painel já aberto → fechar (toggle)
        close_panel()
    else:
        # Abrir painel lateral (etapa 1: caixa ASPEKT / aceitar / rejeitar)
        show_panel(idx, pos)
        render_paper()  # atualiza badge

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_select.on_click(on_select)

# ── Captura de teclas (setas + Y) ────────────────────────────
keyboard_js = widgets.HTML("""
<script>
(function() {
  if (window._aspektListenerAttached) return;
  window._aspektListenerAttached = true;
  document.addEventListener('keydown', function(e) {
    if (['INPUT','TEXTAREA','SELECT'].includes(document.activeElement.tagName)) return;
    var btns = Array.from(document.querySelectorAll('button.widget-button'));
    if (e.key === 'ArrowRight') {
      var b = btns.find(b => b.textContent.includes('Próximo'));
      if (b) b.click();
    } else if (e.key === 'ArrowLeft') {
      var b = btns.find(b => b.textContent.includes('Anterior'));
      if (b) b.click();
    } else if (e.key === 'y' || e.key === 'Y') {
      var b = btns.find(b => b.textContent.includes('Selecionar') || b.textContent.includes('Desmarcar'));
      if (b) b.click();
    }
  });
})();
</script>
""")

# ── Montagem final ───────────────────────────────────────────
sep = widgets.HTML('<hr style="border:none;border-top:1px solid #e2e8f0;margin:4px 0">')

ui = widgets.VBox([
    keyboard_js,
    nav_bar,
    sep,
    body_row,
    sep,
    widgets.HTML('<span style="font-size:11px;color:#94a3b8">Log de ações:</span>'),
    log_out,
])

render_paper()
display(ui)

### **Análise da Sessão**

In [ ]:
# Diagnóstico geral do JSON de busca
total = len(papers)
sem_doi  = sum(1 for p in papers if not p.get("doi"))
sem_urls = sum(1 for p in papers if not p.get("urls"))
sem_ambos = sum(1 for p in papers if not p.get("doi") and not p.get("urls"))

print(f"Total de artigos : {total}")
print(f"Sem DOI          : {sem_doi}")
print(f"Sem URLs         : {sem_urls}")
print(f"Sem DOI e URLs   : {sem_ambos}")
print()

# Mostra as chaves disponíveis no primeiro artigo (para ver a estrutura real)
print("Chaves do 1º artigo:", list(papers[0].keys()))
print()

# Mostra os primeiros 3 artigos completos
for i, p in enumerate(papers[:3]):
    print(f"--- Artigo #{i} ---")
    for k, v in p.items():
        print(f"  {k}: {v}")
    print()



In [ ]:
# ============================================================
# RESUMO DA SESSÃO 
# ============================================================

print(f"Total de artigos: {len(papers)}")
print(f"Artigos selecionados: {len(selected)}")
print()
if selected:
    from collections import Counter
    counts = Counter(selected.values())
    print("Distribuição por caixa ASPEKT:")
    for box, n in sorted(counts.items()):
        print(f"  {box}: {n}")
else:
    print("Nenhum artigo selecionado ainda.")